In [20]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ucphm import *
from functions.ct_column import *
from functions.rsf import *
from scipy.stats import skew, shapiro
from sklearn.preprocessing import OneHotEncoder

In [21]:
dataset = pd.read_excel('../../BreastCareCond.xlsx')
data_code = pd.read_excel('../../Codificação variaveis.xlsx')

Using the variables she used

In [22]:
#Defining the proposed has good treatment variables to use (the empty ones were removed)
relevant_variables = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC']
numerical_variables = ['Age', 'Ki67 positive cells (%) at CB']
categorical_variables = ['N (Regional nodes affected)', 'Number identifying the lesion in the patient', 
                         'Side location of the lesion', 'Classification with respect to other lesions', 
                         'Source of referral', 'Type of invasion at CB', 'Istotype at CB', 'Grade at CB', 
                         'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 
                         'Her2 overexpression (with immunohystochemistry) at CB', 'Disease extent']
date_columns = ['Date of CB ', 'Date of 1st breast operation', 'Date of intervention on sentinel lymph node',
                'Date of intervention (axilla)']
selected_variables = relevant_variables + date_columns+ numerical_variables + categorical_variables

Replace missing values first

In [23]:
dataset = filter_EBC(dataset)
dataset = dataset.reset_index(drop=True)

In [24]:
missing_values(dataset, categorical_variables)

,Column,Missing Values,Percentage Missing
0,N (Regional nodes affected),14,0.383772
1,Number identifying the lesion in the patient,0,0.000000
2,Side location of the lesion,18,0.493421
3,Classification with respect to other lesions,12,0.328947
4,Source of referral,0,0.000000
5,Type of invasion at CB,159,4.358553
6,Istotype at CB,159,4.358553
7,Grade at CB,221,6.058114
8,Oestrogen receptor status at CB,154,4.221491
9,Progesterone receptor status at CB,521,14.281798


Creating the cT variable like she did

In [25]:
grouped_T(dataset)
cT (dataset)
clinical_staging (dataset)
new_variables = ['cT']
selected_variables = selected_variables + new_variables
categorical_variables = categorical_variables + new_variables

In [26]:
change_to_numeric(dataset, numerical_variables)
change_to_categorical(dataset, categorical_variables)

Done
Done


In [27]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True, drop=None)
ohe_input = encoder.fit_transform(dataset[categorical_variables])

dense_matrix = ohe_input.toarray()

# get feature names back out of the encoder
feature_names = encoder.get_feature_names_out(categorical_variables)

# and rebuild as a dataframe
encoded_df = pd.DataFrame(dense_matrix, columns=feature_names, index=dataset.index)
df_clean = encoded_df.loc[:, ~encoded_df.columns.str.endswith('nan')]
feature_names = [col for col in feature_names if not col.endswith('nan')]

In [28]:
ohe_dataset = pd.concat([dataset, df_clean], axis=1)

In [29]:
ohe_dataset = date_of_diagnosis(ohe_dataset, date_columns)

In [30]:
selected_variables = [col for col in selected_variables if col not in categorical_variables]
selected_variables = [col for col in selected_variables if col not in date_columns]
selected_variables = selected_variables + feature_names
date_of_diagnosis = ['Date of diagnosis']
selected_variables = selected_variables + date_of_diagnosis

categorical_variables = feature_names

survival_dataset = ohe_dataset[selected_variables].copy()

In [31]:
median_imputer(survival_dataset, numerical_variables)

'Done'

In [32]:
with open("numerical.txt", "w") as file:
    for item in numerical_variables:
        file.write(str(item) + "\n")
with open("categorical.txt", "w") as file:
    for item in categorical_variables:
        file.write(str(item) + "\n")

In [33]:
relevant_dates = ['Date of diagnosis', 'Date of 1st regional recurrence']
survival_dataset = remove_zeros_na(survival_dataset, relevant_dates)
survival_dataset = format_date(survival_dataset, relevant_dates)

In [34]:
survival_dataset = survival_days(survival_dataset, ['Date of diagnosis'], ['Date of 1st regional recurrence'])
survival_dataset = tte_negative_patients(survival_dataset)
survival_dataset.reset_index(drop=True, inplace=True)

In [35]:
survival_dataset.to_csv("data_treated.csv")